In [1]:
"""
seleccionar_palabras_juez.py
=============================

Arma la lista de palabras a evaluar con el juez LLM (llm_juez_local.py),
combinando tres fuentes para que la muestra sea informativa y no solo
aleatoria:

  1. Las palabras YA revisadas manualmente a lo largo del estudio -- para
     estas ya existe una "verdad" independiente (tu propia lectura de los
     tweets), así que sirven para calibrar si el juez LLM coincide con tu
     criterio antes de confiar en él para el resto.
  2. Las palabras de MAYOR desacuerdo entre los 5 modelos de clusterización
     (concordancia_top_desacuerdo.csv) -- los casos más informativos para
     decidir si el desacuerdo refleja polisemia real o ruido.
  3. Una muestra de palabras de BAJO desacuerdo / alto consenso entre modelos
     -- para no sesgar la evaluación solo hacia los casos difíciles (así se
     descubrió el hallazgo de "acuerdo global ≠ validez" con
     carita/populista/acabo/homenaje).

Salida: CSV con la lista final (palabra, fuente(s), n_ocurrencias si está
disponible) y una lista de Python lista para copiar/pegar en el notebook del
juez.

Requiere en el directorio de trabajo:
  - concordancia_top_desacuerdo.csv       (de calcular_concordancia.py)
  - resultados_clustering_v2.csv          (de clusterizar_sentidos.py)
"""

import pandas as pd

In [16]:
# ---------------------------------------------------------------------------
# Configuración
# ---------------------------------------------------------------------------

RUTA_DESACUERDO = "concordancia_top_desacuerdo.csv"
RUTA_CLUSTERING = "resultados_clustering_v2.csv"
RUTA_SALIDA = "palabras_para_juez.csv"

N_DESACUERDO = 60     # cuántas palabras tomar del top de mayor desacuerdo
N_CONSENSO = 50        # cuántas palabras de bajo desacuerdo / alto consenso agregar
MIN_OCURRENCIAS = 20   # mínimo de ocurrencias en el corpus para que valga la pena
                       # mandarla al juez (necesita variedad de tweets para poder
                       # detectar más de un sentido si existe)

# Palabras ya revisadas manualmente en las distintas "tandas" de inspección a
# lo largo del estudio -- ver protocolo, sección de revisión manual.
PALABRAS_REVISADAS_MANUALMENTE = [
    "elefante", "banco", "gato",  # controles originales (calibración inicial)
    "torres", "entrada", "pagar", "jugar",                    # tanda 1
    "verde", "fuerza", "condiciones", "viaje", "california",  # tanda 2
    "saltillo", "lucha", "temporada", "rico", "juan",         # tanda 3
    "busca", "vida", "esperar", "leer", "confianza", "chido", "entiendo",  # tanda 4
    "carita", "populista", "acabo", "homenaje",               # tanda 5 (acuerdo global)
]

In [17]:
def _normalizar_lista(palabras):
    return sorted(set(p.strip().lower() for p in palabras if p and p.strip()))


def cargar_top_desacuerdo(ruta=RUTA_DESACUERDO, n=N_DESACUERDO):
    df = pd.read_csv(ruta)
    col_palabra = "word_normalized" if "word_normalized" in df.columns else df.columns[0]
    return _normalizar_lista(df[col_palabra].head(n).tolist())


def cargar_consenso(ruta=RUTA_CLUSTERING, n=N_CONSENSO, min_ocurrencias=MIN_OCURRENCIAS):
    """Palabras donde los modelos con k_clusters válido (no NaN) están de
    acuerdo (mismo k, o a lo sumo 1 de diferencia), con suficientes
    ocurrencias en el corpus para que la comparación con el juez tenga
    sentido. Se ordenan por n_ocurrencias descendente y se toman las top n,
    para priorizar palabras con más contexto disponible."""
    df = pd.read_csv(ruta)
    col_palabra = "word_normalized" if "word_normalized" in df.columns else "palabra"

    pivot = df.pivot_table(index=col_palabra, columns="modelo", values="k_clusters")
    desacuerdo = pivot.max(axis=1) - pivot.min(axis=1)
    n_modelos_validos = pivot.notna().sum(axis=1)

    candidatas = desacuerdo[(desacuerdo <= 1) & (n_modelos_validos == pivot.shape[1])].index

    if "n_ocurrencias" in df.columns:
        n_ocurrencias = df.groupby(col_palabra)["n_ocurrencias"].max()
        candidatas = [p for p in candidatas if n_ocurrencias.get(p, 0) >= min_ocurrencias]
        candidatas = sorted(candidatas, key=lambda p: n_ocurrencias.get(p, 0), reverse=True)
    else:
        candidatas = list(candidatas)

    return _normalizar_lista(candidatas[:n])


def construir_lista_final():
    manuales = _normalizar_lista(PALABRAS_REVISADAS_MANUALMENTE)
    try:
        desacuerdo = cargar_top_desacuerdo()
    except FileNotFoundError:
        print(f"Aviso: no se encontró {RUTA_DESACUERDO}, se omite esa fuente.")
        desacuerdo = []
    try:
        consenso = cargar_consenso()
    except FileNotFoundError:
        print(f"Aviso: no se encontró {RUTA_CLUSTERING}, se omite esa fuente.")
        consenso = []

    filas = []
    todas = set(manuales) | set(desacuerdo) | set(consenso)
    for palabra in sorted(todas):
        fuentes = []
        if palabra in manuales:
            fuentes.append("revisada_manualmente")
        if palabra in desacuerdo:
            fuentes.append("alto_desacuerdo")
        if palabra in consenso:
            fuentes.append("consenso")
        filas.append({"palabra": palabra, "fuentes": ",".join(fuentes)})

    resultado = pd.DataFrame(filas).sort_values("palabra").reset_index(drop=True)
    return resultado, manuales, desacuerdo, consenso

In [18]:

resultado, manuales, desacuerdo, consenso = construir_lista_final()

resultado.to_csv(RUTA_SALIDA, index=False)

print(f"Palabras revisadas manualmente: {len(manuales)}")
print(f"Palabras de mayor desacuerdo (top {N_DESACUERDO}): {len(desacuerdo)}")
print(f"Palabras de consenso (bajo desacuerdo, top {N_CONSENSO}): {len(consenso)}")
print(f"Total combinado (sin duplicados): {len(resultado)}")
print(f"\nGuardado en: {RUTA_SALIDA}\n")

lista_python = resultado["palabra"].tolist()
print("Lista lista para pegar en el notebook del juez:")
print(lista_python)






Palabras revisadas manualmente: 28
Palabras de mayor desacuerdo (top 60): 20
Palabras de consenso (bajo desacuerdo, top 50): 50
Total combinado (sin duplicados): 84

Guardado en: palabras_para_juez.csv

Lista lista para pegar en el notebook del juez:
['acabo', 'ahorita', 'ando', 'banco', 'busca', 'california', 'carita', 'chido', 'ciudad', 'clase', 'completa', 'condiciones', 'confianza', 'covid', 'darle', 'desarrollo', 'deseo', 'diga', 'dolor', 'duele', 'elefante', 'entiendo', 'entrada', 'equipo', 'escuchar', 'escuela', 'espacio', 'esperar', 'federal', 'frente', 'fuerza', 'gato', 'gobernador', 'gustan', 'gusto', 'hablando', 'hablar', 'hice', 'homenaje', 'imagen', 'jaja', 'juan', 'jugar', 'leer', 'llega', 'lucha', 'mano', 'miedo', 'millones', 'municipal', 'municipio', 'nivel', 'oportunidad', 'pagar', 'palabras', 'pendeja', 'personal', 'plan', 'populista', 'precio', 'presidente', 'rico', 'sacar', 'salir', 'saltillo', 'seguir', 'siguen', 'sistema', 'tanta', 'tardes', 'temporada', 'tengan',